In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResNetBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size= 3, padding= 1)
        self.norm1 = nn.GroupNorm(8, channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size= 3, padding= 1)
        self.norm2 = nn.GroupNorm(8, channels)

    def forward(self, x):
        res = x
        x = F.silu(self.norm1(self.conv1(x)))
        x = self.norm2(self.conv2(x))
        return F.silu(x + res)

In [2]:
class VAE(nn.Module):
    def __init__(self, in_channels= 3, latent_channels= 4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size= 4, stride= 2, padding= 1),
            nn.ReLU(),
            
            nn.Conv2d(64, 128, kernel_size= 4, stride= 2, padding= 1),
            nn.ReLU(),
            
            nn.Conv2d(128, 256, kernel_size= 4, stride= 2, padding= 1),
            nn.ReLU(),

            ResNetBlock(256),
            nn.Conv2d(256, latent_channels * 2, kernel_size= 3, padding= 1)
        )

        self.decoder = nn.Sequential(
            nn.Conv2d(latent_channels, 256, kernel_size= 3, padding= 1),
            ResNetBlock(256),

            nn.ConvTranspose2d(256, 128, kernel_size= 4, stride= 2, padding= 1),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, kernel_size= 4, stride= 2, padding= 1),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 32, kernel_size= 4, stride= 2, padding= 1),
            nn.ReLU(),
        
            nn.Conv2d(32, in_channels, kernel_size= 3,padding= 1),
            
            nn.Sigmoid()
        )

    def encode(self, x):
        moments = self.encoder(x)
        mean, logvar = torch.chunk(moments, 2, dim= 1)
        logvar = logvar.clamp(-30.0, 20.0)

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mean + eps * std
        return z, mean, logvar

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z, mean, logvar = self.encode(x)
        recon_x = self.decode(z)
        return recon_x, mean, logvar

In [3]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

class CelebDataset(Dataset):
    def __init__(self, root_dir, transform= None):
        self.root_dir = root_dir
        self.transform = transform

        self.images = os.listdir(root_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        path = os.path.join(self.root_dir, self.images[idx])
        image = Image.open(path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image

transform = transforms.Compose([transforms.Resize((128, 128)), transforms.ToTensor()])

dataset = CelebDataset("data/img_align_celeba", transform= transform)
small_dataset = Subset(dataset, range(5000))

dataloader = DataLoader(small_dataset, batch_size= 8, shuffle= True)

In [8]:
from tqdm import tqdm
from torchvision.utils import save_image

device = "cuda" if torch.cuda.is_available() else "cpu"

model = VAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr= 1e-4)

images = next(iter(dataloader)).to(device)
reconstructed, mean, logvar = model(images)
print(images.shape)
print(reconstructed.shape)
print(mean.shape)

epochs = 10
for epoch in range(epochs):
    model.train()

    total_loss = 0
    total_recon = 0
    total_kl = 0

    progress_bar = tqdm(dataloader)
    for batch_idx, images in enumerate(progress_bar):

        images = images.to(device)
        reconstructed, mean, logvar = model(images)
        recon_loss = F.mse_loss(reconstructed, images)

        kl_loss = -0.5 * torch.mean(1 + logvar - mean.pow(2) - logvar.exp())
        loss = recon_loss + 0.001 * kl_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()

        progress_bar.set_description(f"Epoch {epoch+1}/{epochs}")
        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "recon": f"{recon_loss.item():.4f}",
            "kl": f"{kl_loss.item():.4f}"
        })

    avg_loss = total_loss / len(dataloader)
    avg_recon = total_recon / len(dataloader)
    avg_kl = total_kl / len(dataloader)

    print(f"Epoch {epoch+1}")
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Average Recon: {avg_recon:.4f}")
    print(f"Average KL: {avg_kl:.4f}")

    with torch.no_grad():
        sample = torch.cat([images[:8], reconstructed[:8]])
        save_image(sample, f"images/results_vae_epoch_{epoch+1}.png", nrow= 8)

torch.Size([8, 3, 128, 128])
torch.Size([8, 3, 128, 128])
torch.Size([8, 4, 16, 16])


Epoch 1/10: 100%|██████████| 625/625 [00:53<00:00, 11.67it/s, loss=0.0069, recon=0.0058, kl=1.1684]




Epoch 1
Average Loss: 0.0170
Average Recon: 0.0159
Average KL: 1.1085


Epoch 2/10: 100%|██████████| 625/625 [00:52<00:00, 11.81it/s, loss=0.0045, recon=0.0034, kl=1.1451]




Epoch 2
Average Loss: 0.0063
Average Recon: 0.0051
Average KL: 1.1891


...



Epoch 8
Average Loss: 0.0044
Average Recon: 0.0031
Average KL: 1.2856


Epoch 9/10: 100%|██████████| 625/625 [00:54<00:00, 11.41it/s, loss=0.0035, recon=0.0023, kl=1.2373]




Epoch 9
Average Loss: 0.0041
Average Recon: 0.0028
Average KL: 1.3167


Epoch 10/10: 100%|██████████| 625/625 [00:52<00:00, 11.91it/s, loss=0.0037, recon=0.0024, kl=1.2973]




Epoch 10
Average Loss: 0.0041
Average Recon: 0.0028
Average KL: 1.3298


In [9]:
torch.save(model.state_dict(), "models/vae.pth")

In [11]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = VAE().to(device)
model.load_state_dict(torch.load("models/vae.pth", map_location= device))
model.eval()

print("Model loaded successfully")

Model loaded successfully


In [14]:
images = next(iter(dataloader)).to(device)

with torch.no_grad():
    reconstructed, mean, logvar = model(images)

print(reconstructed.shape) # torch.Size([8, 3, 128, 128])

torch.Size([8, 3, 128, 128])
